# Agent 1: Resume Extractor Agent

This notebook implements the complete, detailed implementation of the **Resume Extractor Agent** used in CareerAtlas. The agent parses resume PDFs, converts them to Markdown/text, and uses a structured LLM call with a repair loop to extract a rich, standardized profile matching the database schema.


### Step 1: API Keys Setup

Please configure your API keys here.


In [ ]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

class Settings:
    google_api_key = os.environ.get("GOOGLE_API_KEY", "")
    google_api_keys = [google_api_key] if google_api_key else []

settings = Settings()

In [ ]:
# Install dependencies
# !pip install PyMuPDF pymupdf4llm phonenumbers pydantic langchain-google-genai

### Step 2: Imports & Pydantic Schemas

Exact schema definitions from `app.resume_extraction.schemas`.


In [ ]:
import json
import re
import tempfile
import time
from typing import Any, Optional, List
from urllib.parse import urlparse
import fitz  # PyMuPDF
import phonenumbers
import pymupdf4llm
from phonenumbers import PhoneNumberFormat
from pydantic import BaseModel, Field, ValidationError
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(BaseModel):
    email: Optional[str] = Field(default=None, description="Primary email address.")
    phone_raw: Optional[str] = Field(default=None, description="Raw phone number as extracted.")
    country_code: Optional[str] = Field(default=None, description="Country code (e.g., +91).")
    national_number: Optional[str] = Field(default=None, description="National number without country code.")
    e164_phone: Optional[str] = Field(default=None, description="E.164 formatted phone number.")
    location: Optional[str] = Field(default=None, description="Location such as city, state, or country.")
    linkedin: Optional[str] = Field(default=None, description="LinkedIn profile URL.")
    github: Optional[str] = Field(default=None, description="GitHub profile URL.")
    website: Optional[str] = Field(default=None, description="Personal website or portfolio URL.")

class ExperienceItem(BaseModel):
    company: Optional[str] = Field(default=None, description="Company or organization name.")
    title: Optional[str] = Field(default=None, description="Job title or role.")
    location: Optional[str] = Field(default=None, description="Job location if present.")
    start_date: Optional[str] = Field(default=None, description="Start date as written in the resume.")
    end_date: Optional[str] = Field(default=None, description="End date as written in the resume.")
    is_current: Optional[bool] = Field(default=None, description="True if this is the current role.")
    description_bullets: List[str] = Field(default_factory=list, description="Resume bullet points for the role.")
    technologies: List[str] = Field(default_factory=list, description="Technologies, tools, or methods mentioned for the role.")

class EducationItem(BaseModel):
    institution: Optional[str] = Field(default=None, description="School, college, or university name.")
    degree: Optional[str] = Field(default=None, description="Degree or qualification.")
    field_of_study: Optional[str] = Field(default=None, description="Branch, major, or specialization.")
    start_date: Optional[str] = Field(default=None, description="Start date if present.")
    end_date: Optional[str] = Field(default=None, description="End date if present.")
    grade: Optional[str] = Field(default=None, description="CGPA, percentage, or grade if present.")
    notes: List[str] = Field(default_factory=list, description="Additional notes such as coursework.")

class ProjectItem(BaseModel):
    name: Optional[str] = Field(default=None, description="Project name.")
    description: Optional[str] = Field(default=None, description="Short description of the project.")
    technologies: List[str] = Field(default_factory=list, description="Tools, libraries, frameworks, or languages used.")
    link: Optional[str] = Field(default=None, description="Project URL if present.")

class CertificationItem(BaseModel):
    name: Optional[str] = Field(default=None, description="Certification name.")
    issuer: Optional[str] = Field(default=None, description="Issuing organization.")
    date: Optional[str] = Field(default=None, description="Date if present.")
    credential_id: Optional[str] = Field(default=None, description="Credential ID if present.")
    link: Optional[str] = Field(default=None, description="Credential URL if present.")

class ResumeExtraction(BaseModel):
    full_name: Optional[str] = Field(default=None, description="Candidate full name.")
    headline: Optional[str] = Field(default=None, description="Professional headline.")
    contact: ContactInfo
    summary: Optional[str] = Field(default=None, description="Professional summary.")
    skills: List[str] = Field(default_factory=list, description="All professional skills.")
    programming_languages: List[str] = Field(default_factory=list, description="Programming languages.")
    spoken_languages: List[str] = Field(default_factory=list, description="Human languages.")
    experience: List[ExperienceItem] = Field(default_factory=list)
    education: List[EducationItem] = Field(default_factory=list)
    projects: List[ProjectItem] = Field(default_factory=list)
    certifications: List[CertificationItem] = Field(default_factory=list)
    keywords: List[str] = Field(default_factory=list, description="Important keywords.")

### Step 3: LLM Factory & Rotation Logic

Exact rotating Gemini invocation logic from `app.utils.llm_factory`.


In [ ]:
def _get_rotating_gemini_model(model_name: str, temperature: float, attempt: int):
    keys = settings.google_api_keys
    if not keys:
        raise ValueError("No GOOGLE_API_KEY available for Gemini.")
    key = keys[attempt % len(keys)]
    return ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=key,
        temperature=temperature,
        max_retries=0,
        timeout=60.0
    )

def invoke_gemini(prompt: Any, model_name: str = "gemini-2.5-flash", temperature: float = 0.2, schema: Any = None, max_retries: int = 5):
    last_exception = None
    for attempt in range(max_retries):
        try:
            model = _get_rotating_gemini_model(model_name, temperature, attempt)
            chain = model.with_structured_output(schema) if schema else model
            if hasattr(prompt, "invoke") and hasattr(prompt, "|"):
                return (prompt | chain).invoke({})
            else:
                return chain.invoke(prompt)
        except Exception as e:
            last_exception = e
            print(f"Gemini API Error on attempt {attempt+1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    raise last_exception

### Step 4: URL, Phone and Skill Parsing Logic

Exact normalization functions from `app.resume_extraction.service`.


In [ ]:
URL_TRAILING_PUNCT = ".,;:!?)]}>'\""
KNOWN_RESUME_TLDS = {"com", "in", "io", "dev", "net", "org", "ai", "co", "me", "app", "tech", "info", "edu", "gov"}
COMMON_SKILL_TERMS = [
    "Python", "Java", "JavaScript", "TypeScript", "C++", "C#", "C", "Go", "Rust", "SQL",
    "HTML", "CSS", "React", "Next.js", "Node.js", "FastAPI", "Django", "Flask",
    "Pandas", "NumPy", "scikit-learn", "TensorFlow", "PyTorch", "AWS", "Docker", "Kubernetes"
]

def normalize_phone(raw_phone: str | None, default_region: str = "IN") -> dict:
    if not raw_phone: return {"phone_raw": None, "country_code": None, "national_number": None, "e164_phone": None}
    try:
        parsed = phonenumbers.parse(raw_phone, default_region)
        if not phonenumbers.is_valid_number(parsed):
            return {"phone_raw": raw_phone, "country_code": None, "national_number": None, "e164_phone": None}
        return {
            "phone_raw": raw_phone,
            "country_code": f"+{parsed.country_code}",
            "national_number": str(parsed.national_number),
            "e164_phone": phonenumbers.format_number(parsed, PhoneNumberFormat.E164),
        }
    except Exception:
        return {"phone_raw": raw_phone, "country_code": None, "national_number": None, "e164_phone": None}

def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

def strip_trailing_punctuation(url: str) -> str:
    url = url.strip()
    while url and url[-1] in URL_TRAILING_PUNCT:
        url = url[:-1]
    return url

def normalize_url(url: str | None) -> str | None:
    if not url: return None
    url = strip_trailing_punctuation(url.strip())
    if url.startswith("<") and url.endswith(">"):
        url = url[1:-1].strip()
    if url.lower().startswith("mailto:"): return None
    parsed = urlparse(url)
    candidate = None
    if parsed.scheme in {"http", "https"} and parsed.netloc: candidate = url
    elif re.match(r"^(www\.)?([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}(/.*)?$", url): candidate = "https://" + url
    if not candidate: return None
    parsed_candidate = urlparse(candidate)
    host = (parsed_candidate.netloc or "").split(":")[0]
    if "." not in host: return None
    tld = host.rsplit(".", 1)[-1].lower()
    if tld not in KNOWN_RESUME_TLDS: return None
    return candidate

def extract_urls_from_text(text: str) -> List[str]:
    text = text or ""
    pattern = re.compile(r"(?ix)(?<!@)\b(https?://[^\s<>()\[\]{}"']+|www\.[^\s<>()\[\]{}"']+|[a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:/[^\s<>()\[\]{}"']*)?)")
    found = []
    for match in pattern.finditer(text):
        url = normalize_url(match.group(1))
        if url: found.append(url)
    return list(dict.fromkeys(found))

def extract_embedded_urls(pdf_path: str) -> List[str]:
    try:
        doc = fitz.open(pdf_path)
        urls = []
        for page in doc:
            for link in page.get_links():
                uri = normalize_url(link.get("uri"))
                if uri: urls.append(uri)
        return list(dict.fromkeys(urls))
    except Exception:
        return []

def classify_url(url: str) -> str:
    u = url.lower()
    if "linkedin.com" in u: return "contact.linkedin"
    if "github.com" in u: return "contact.github"
    if "twitter.com" in u or "x.com" in u: return "contact.twitter"
    if "leetcode.com" in u: return "contact.leetcode"
    if "kaggle.com" in u: return "contact.kaggle"
    return "other"

def build_url_manifest(urls: List[str]) -> str:
    manifest = []
    for url in urls:
        parsed = urlparse(url)
        manifest.append({"kind": classify_url(url), "url": url, "domain": parsed.netloc.lower() if parsed.netloc else ""})
    return json.dumps(manifest, ensure_ascii=False, indent=2)

def extract_pdf_text(pdf_path: str) -> str:
    try:
        text = pymupdf4llm.to_markdown(pdf_path, header=False, footer=False)
        text = normalize_text(text) if text and text.strip() else ""
    except Exception:
        text = ""
    if not text:
        doc = fitz.open(pdf_path)
        parts = [page.get_text("text", sort=True) for page in doc]
        text = normalize_text("\n\n".join(parts))
    embedded_urls = extract_embedded_urls(pdf_path)
    visible_urls = extract_urls_from_text(text)
    all_urls = []
    seen = set()
    for url in embedded_urls + visible_urls:
        normalized = normalize_url(url)
        if normalized and normalized not in seen:
            seen.add(normalized)
            all_urls.append(normalized)
    if all_urls:
        text += "\n\n[EXTRACTED_URLS_JSON]\n" + build_url_manifest(all_urls) + "\n[/EXTRACTED_URLS_JSON]"
    return text

def pdf_bytes_to_markdown(pdf_bytes: bytes) -> str:
    temp_file = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    temp_path = temp_file.name
    try:
        temp_file.write(pdf_bytes)
        temp_file.close()
        return extract_pdf_text(temp_path)
    finally:
        try: os.remove(temp_path)
        except OSError: pass

### Step 5: Post-Processing & Parsing Pipeline

Handles nested schema parsing and repair logic.


In [ ]:
def _strip_code_fences(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text, flags=re.IGNORECASE)
    return text.strip()

def _dedupe_strings(values: List[str]) -> List[str]:
    seen = set()
    out = []
    for value in values:
        text = (value or "").strip()
        if text and text not in seen:
            seen.add(text)
            out.append(text)
    return out

def _infer_skills_from_text(resume_text: str) -> List[str]:
    text = resume_text or ""
    found = []
    lowered = text.lower()
    for term in COMMON_SKILL_TERMS:
        idx = lowered.find(term.lower())
        if idx != -1: found.append((idx, term))
    found.sort(key=lambda item: item[0])
    return _dedupe_strings([term for _, term in found])

def _stringify_list(value: Any) -> List[str]:
    items = []
    seen = set()
    def add(item: Any) -> None:
        if item is None: return
        if isinstance(item, str):
            chunks = re.split(r"[,\n;|/•·]+", item)
            if len(chunks) > 1:
                for chunk in chunks: add(chunk)
                return
            text = item.strip()
            if text and text not in seen:
                seen.add(text)
                items.append(text)
        elif isinstance(item, list):
            for sub in item: add(sub)
        elif isinstance(item, dict):
            for key in ("name", "title", "skill", "value"):
                val = item.get(key)
                if isinstance(val, str) and val.strip(): add(val)
    add(value)
    return items

def _normalize_blocks(blocks: Any) -> List[dict]:
    normalized = []
    for block in blocks or []:
        if not isinstance(block, dict): continue
        item = dict(block)
        item["description_bullets"] = _stringify_list(item.get("description_bullets") or item.get("bullets") or [])
        item["technologies"] = _stringify_list(item.get("technologies") or item.get("tech") or [])
        normalized.append(item)
    return normalized

def _normalize_education(blocks: Any) -> List[dict]:
    normalized = []
    for block in blocks or []:
        if not isinstance(block, dict): continue
        item = dict(block)
        item["notes"] = _stringify_list(item.get("notes") or item.get("coursework") or [])
        normalized.append(item)
    return normalized

def _normalize_projects(blocks: Any) -> List[dict]:
    normalized = []
    for block in blocks or []:
        if not isinstance(block, dict): continue
        item = dict(block)
        if not item.get("name"): item["name"] = item.get("title") or item.get("project_name")
        item["technologies"] = _stringify_list(item.get("technologies") or item.get("tech") or [])
        normalized.append(item)
    return normalized

def _enforce_post_processing(parsed: ResumeExtraction) -> ResumeExtraction:
    parsed_dict = parsed.model_dump(mode="json")
    contact = parsed_dict.get("contact", {}) or {}
    contact.update(normalize_phone(contact.get("phone_raw")))
    for field in ("linkedin", "github", "website"): contact[field] = normalize_url(contact.get(field))
    parsed_dict["contact"] = contact
    return ResumeExtraction.model_validate(parsed_dict)

def _backfill_skills(parsed: ResumeExtraction, resume_text: str) -> ResumeExtraction:
    data = parsed.model_dump(mode="json")
    skills = _dedupe_strings(list(data.get("skills") or []) + list(data.get("programming_languages") or []) + list(data.get("keywords") or []))
    for exp in data.get("experience", []) or []:
        if isinstance(exp, dict): skills.extend(_dedupe_strings(list(exp.get("technologies") or [])))
    for proj in data.get("projects", []) or []:
        if isinstance(proj, dict): skills.extend(_dedupe_strings(list(proj.get("technologies") or [])))
    skills = _dedupe_strings(skills)
    if not skills: skills = _infer_skills_from_text(resume_text)
    data["skills"] = skills
    return ResumeExtraction.model_validate(data)

### Step 6: Main Pipeline & Repair Loop

Binds the prompts, Gemini structured output invocation, and parsing helpers together.


In [ ]:
def build_extraction_prompt(resume_text: str) -> str:
    schema_json = json.dumps(ResumeExtraction.model_json_schema(), indent=2)
    return f"""You are a precise resume information extraction engine.
Return ONLY valid JSON matching the provided schema.

--- TARGET JSON SCHEMA ---
{schema_json}

Resume text:
{resume_text}
""".strip()

def build_repair_prompt(resume_text: str, draft_json: str, validation_error: str) -> str:
    schema_json = json.dumps(ResumeExtraction.model_json_schema(), indent=2)
    return f"""You produced JSON that failed schema validation. Fix it.
Validation error:
{validation_error}
Previous JSON:
{draft_json}
""".strip()

def parse_resume_json(candidate_json: str) -> ResumeExtraction:
    raw = json.loads(_strip_code_fences(candidate_json))
    raw["skills"] = _stringify_list(raw.get("skills", []))
    raw["programming_languages"] = _stringify_list(raw.get("programming_languages", []))
    raw["spoken_languages"] = _stringify_list(raw.get("spoken_languages", []))
    raw["keywords"] = _stringify_list(raw.get("keywords", []))
    raw["experience"] = _normalize_blocks(raw.get("experience", []))
    raw["education"] = _normalize_education(raw.get("education", []))
    raw["projects"] = _normalize_projects(raw.get("projects", []))
    return ResumeExtraction.model_validate(raw)

def extract_structured_resume_data(md_text: str) -> ResumeExtraction:
    prompt = build_extraction_prompt(md_text)
    try:
        draft_json = invoke_gemini(prompt, temperature=0.0)
        # If the returned object is a ChatMessage, read content
        if hasattr(draft_json, "content"):
            draft_json = draft_json.content
    except Exception as e:
        raise RuntimeError(f"Gemini API call failed: {e}")
        
    for attempt in range(2):
        try:
            parsed = parse_resume_json(draft_json)
            parsed = _enforce_post_processing(parsed)
            parsed = _backfill_skills(parsed, md_text)
            return parsed
        except (ValidationError, json.JSONDecodeError) as exc:
            if attempt == 0:
                print(f"Validation failed. Retrying repair... Error: {exc}")
                repair_prompt = build_repair_prompt(md_text, draft_json, str(exc))
                draft_json = invoke_gemini(repair_prompt, temperature=0.0)
                if hasattr(draft_json, "content"): draft_json = draft_json.content
            else:
                raise

### Step 7: Verification Example

Run this with a set `GOOGLE_API_KEY` env variable to extract structured data.


In [ ]:
sample_resume = """
Jane Doe
jane.doe@example.com | +91 9999999999
github.com/janedoe | linkedin.com/in/janedoe

Software Engineer with experience in Python, AWS and Docker.

Experience:
- Software Developer at XYZ Corp (Jan 2023 - Present):
  * Built REST APIs using FastAPI and Python.
  * Configured AWS ECS tasks using Docker.
"""

try:
    extracted_data = extract_structured_resume_data(sample_resume)
    print("Successfully Extracted Resume Profile:")
    print(extracted_data.model_dump_json(indent=2))
except Exception as e:
    print(f"Execution skipped or failed. Error: {e}")